# FairFace Model Evaluation on MSKCC Dataset

This notebook evaluates the performance of the **FairFace-trained EfficientNet-B4** model on the **MSKCC skin tone dataset**.

**Objective:** Assess generalization to clinical close-up images.
**Modes:** Togglable between **3-Way (Light/Medium/Dark)** and **6-Way (I-VI)** evaluation.
**Filter:** Excludes dermoscopic images.

In [ ]:
# ==================== CELL 1: Configuration ====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# --- CONFIGURATION ---
# Set EVAL_MODE to '3-way' or '6-way'
EVAL_MODE = '3-way' 

IMAGE_SIZE = 380
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "outputs/fairface_3way/best_finetuned_model.pth" 
METADATA_PATH = "datasets/mskcc-skin-tone-labeling-dataset_metadata_2025-11-24.csv"
IMAGE_ROOT = "datasets/MSKCC-images"

print(f"Evaluation Mode: {EVAL_MODE}")
print(f"Using device: {DEVICE}")

In [ ]:
# ==================== CELL 2: Data Preparation ====================

print("Loading metadata...")
df = pd.read_csv(METADATA_PATH)

# 1. Filter for Close-up Only
df_filtered = df[df['image_type'] != 'dermoscopic'].copy()
df_filtered = df_filtered.dropna(subset=['fitzpatrick_skin_type'])
print(f"Filtered images: {len(df_filtered)}")

# 2. Define Class Mappings
if EVAL_MODE == '3-way':
    # Ground Truth Mapping
    def map_gt(skin_type):
        if skin_type in ['I', 'II']: return 'Light'
        if skin_type in ['III', 'IV']: return 'Medium'
        if skin_type in ['V', 'VI']: return 'Dark'
        return None
    
    CLASSES = ['Light', 'Medium', 'Dark']
    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
    
    # Mappings for Probabilities
    def aggregate_probs(probs_6way): # probs_6way shape (B, 6)
        # Result shape (B, 3) -> Light, Medium, Dark
        # FairFace Train Output: 0=VI, 1=V, 2=IV, 3=III, 4=II, 5=I
        light = probs_6way[:, 4] + probs_6way[:, 5]
        medium = probs_6way[:, 2] + probs_6way[:, 3]
        dark = probs_6way[:, 0] + probs_6way[:, 1]
        return np.stack([light, medium, dark], axis=1)

else: # 6-way
    def map_gt(skin_type):
        return skin_type
    
    # Order: VI -> I (Top to Bottom)
    CLASSES = ['VI', 'V', 'IV', 'III', 'II', 'I']
    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
    
    def aggregate_probs(probs_6way):
        return probs_6way # No change needed

# Apply GT Mapping
df_filtered['target_label'] = df_filtered['fitzpatrick_skin_type'].apply(map_gt)
df_filtered = df_filtered.dropna(subset=['target_label'])

# Add Paths
df_filtered['path'] = df_filtered['isic_id'].apply(lambda x: os.path.join(IMAGE_ROOT, f"{x}.jpg"))
df_filtered = df_filtered[df_filtered['path'].apply(os.path.exists)]

# RESET INDEX is crucial for matching line numbers later
df_filtered = df_filtered.reset_index(drop=True)

print(f"Final test set size: {len(df_filtered)}")
print(f"Classes: {CLASSES}")
print(df_filtered['target_label'].value_counts())

Loading metadata...
Filtered images: 1201
Final test set size: 1201
Classes: ['Light', 'Medium', 'Dark']
target_label
Light     428
Medium    397
Dark      376
Name: count, dtype: int64


In [ ]:
# ==================== CELL 3: Dataset Class ====================

class InferenceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = df['path'].values
        self.labels = df['target_label'].values
        self.transform = transform
    
    def __len__(self): return len(self.paths)
    
    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except:
            img = Image.new("RGB", (380, 380))
        
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_loader = DataLoader(InferenceDataset(df_filtered, test_transform), 
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
# ==================== CELL 4: Model Loading & Inference ====================

# Always load 6-way model structure
model = models.efficientnet_b4(weights=None)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, 6) 
)

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("✓ Model loaded")
else:
    print("❌ Model not found")

model = model.to(DEVICE).eval()

y_true_labels = []
y_scores = [] # Probabilities

print("Running inference...")
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        
        # Get probabilities (Softmax)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        
        # Transform probs based on mode
        batch_scores = aggregate_probs(probs)
        
        y_scores.extend(batch_scores)
        y_true_labels.extend(labels)

y_scores = np.array(y_scores)
# Convert true labels to indices
y_true_indices = np.array([CLASS_TO_IDX[l] for l in y_true_labels])
# Get predictions
y_pred_indices = np.argmax(y_scores, axis=1)
y_pred_labels = [CLASSES[i] for i in y_pred_indices]

print("✓ Inference complete")

In [ ]:
# ==================== CELL 5: Predict & Evaluate ====================

# 1. Overall Accuracy
accuracy = (y_pred_indices == y_true_indices).mean()
print(f"\n{'='*60}")
print(f"TEST SET ACCURACY: {accuracy:.2%}")
print(f"{'='*60}")

# 2. Classification Report
print("\nDetailed Classification Report:")
print(classification_report(y_true_indices, y_pred_indices, target_names=CLASSES))

# 3. Confusion Matrix
cm = confusion_matrix(y_true_indices, y_pred_indices)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'Confusion Matrix - {EVAL_MODE} Evaluation')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# ==================== CELL 6: Precision-Recall Curves ====================

# Binarize labels for PR curve (one-vs-rest)
y_true_bin = label_binarize(y_true_indices, classes=range(len(CLASSES)))

fig, ax = plt.subplots(figsize=(10, 7))

for i, label in enumerate(CLASSES):
    precision, recall, _ = precision_recall_curve(
        y_true_bin[:, i], 
        y_scores[:, i]
    )
    ap = average_precision_score(y_true_bin[:, i], y_scores[:, i])
    ax.plot(recall, precision, lw=2, label=f'{label} (AP={ap:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'Precision-Recall Curves ({EVAL_MODE})', fontsize=14)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Average Precision Scores:")
for i, label in enumerate(CLASSES):
    ap = average_precision_score(y_true_bin[:, i], y_scores[:, i])
    print(f"  {label:8s}: {ap:.3f}")

In [ ]:
# ==================== CELL 7: Error Analysis with Images ====================

# Find misclassified samples
errors = y_true_indices != y_pred_indices
error_indices = np.where(errors)[0]

print(f"Total errors: {len(error_indices)} out of {len(df_filtered)} ({len(error_indices)/len(df_filtered):.1%})")

# Show first 12 misclassified images
n_to_show = min(12, len(error_indices))
if n_to_show > 0:
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, error_idx in enumerate(error_indices[:n_to_show]):
        # Get image info from ORIGINAL dataframe (indices match due to reset_index)
        img_path = df_filtered.iloc[error_idx]['path']
        true_label = y_true_labels[error_idx]
        pred_label = y_pred_labels[error_idx]
        confidence = y_scores[error_idx].max()
        
        try:
            img = Image.open(img_path)
            axes[idx].imshow(img)
        except:
            pass # Black if missing
            
        axes[idx].axis('off')
        
        # Add title
        title = f"True: {true_label}\nPred: {pred_label} ({confidence:.1%})"
        axes[idx].set_title(title, fontsize=10, color='red', weight='bold')
    
    # Hide unused subplots
    for idx in range(n_to_show, 12):
        axes[idx].axis('off')
    
    plt.suptitle('Misclassified Images', fontsize=16, weight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No errors to display!")

In [ ]:
# ==================== CELL 8: Confusion Analysis per Class ====================

print("\nDetailed Confusion Analysis:\n")

for i, true_label in enumerate(CLASSES):
    # Get samples for this true class
    mask = y_true_indices == i
    true_count = mask.sum()
    
    if true_count == 0: continue
    
    print(f"{true_label} (n={true_count}):")
    
    # Count predictions for this true class
    for j, pred_label in enumerate(CLASSES):
        pred_count = (y_pred_indices[mask] == j).sum()
        percentage = pred_count / true_count * 100
        
        if pred_count > 0:
            symbol = "✓" if i == j else "✗"
            print(f"  {symbol} Predicted as {pred_label:8s}: {pred_count:3d} ({percentage:5.1f}%)")
    print()

In [ ]:
# ==================== CELL 9: Class-wise Accuracy by Fitzpatrick Type ====================

print("Accuracy Breakdown by Original Fitzpatrick Type:\n")

# Create analysis df
analysis_df = df_filtered.copy()
analysis_df['predicted_label'] = y_pred_labels
analysis_df['correct'] = analysis_df['target_label'] == analysis_df['predicted_label']

# Group by original Fitzpatrick type (I, II, III...)
for fitz_type in sorted(analysis_df['fitzpatrick_skin_type'].unique()):
    subset = analysis_df[analysis_df['fitzpatrick_skin_type'] == fitz_type]
    if len(subset) == 0: continue
        
    accuracy = subset['correct'].mean()
    count = len(subset)
    # Show what this maps to in current Mode
    grouped = subset['target_label'].iloc[0]
    
    print(f"Type {fitz_type} (mapped to {grouped}, n={count}): {accuracy:.1%} accuracy")
    
    # Show prediction distribution
    pred_dist = subset['predicted_label'].value_counts()
    for label, cnt in pred_dist.items():
        print(f"  → {label:8s}: {cnt:3d} ({cnt/count*100:5.1f}%)")
    print()